# Day 2: Data Cleaning

This notebook focuses on cleaning various raw data files to ensure consistency for downstream analysis.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os

raw_dir = "../data/raw"
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

## Part 1: Fund Master

Cleaning steps for `01_fund_master.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "01_fund_master.csv"))
df['launch_date'] = pd.to_datetime(df['launch_date'])
df['risk_category'] = df['risk_category'].str.strip().str.title()
# Validate expense ratio range (0-3%)
df = df[(df['expense_ratio_pct'] >= 0) & (df['expense_ratio_pct'] <= 3)]
df.to_csv(os.path.join(processed_dir, "01_fund_master_cleaned.csv"), index=False)
print("Cleaned Fund Master saved.")

## Part 2: NAV History

Cleaning steps for `02_nav_history.csv`.

In [ ]:
nav_df = pd.read_csv(os.path.join(raw_dir, "02_nav_history.csv"))
nav_df['date'] = pd.to_datetime(nav_df['date'])
nav_df = nav_df.sort_values(['amfi_code', 'date'])
nav_df = nav_df.drop_duplicates(subset=['amfi_code', 'date'], keep='last')

def fill_missing_dates(group):
    start_date = group['date'].min()
    end_date = group['date'].max()
    all_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    return group.set_index('date').reindex(all_dates).ffill().reset_index().rename(columns={'index': 'date'})

nav_df = nav_df.groupby('amfi_code', group_keys=False).apply(fill_missing_dates)
nav_df['amfi_code'] = nav_df['amfi_code'].astype(int)
nav_df = nav_df[nav_df['nav'] > 0]
nav_df.to_csv(os.path.join(processed_dir, "02_nav_history_cleaned.csv"), index=False)
print("Cleaned NAV History saved.")

## Part 3: AUM by Fund House

Cleaning steps for `03_aum_by_fund_house.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "03_aum_by_fund_house.csv"))
df['date'] = pd.to_datetime(df['date'])
df = df[(df['aum_crore'] > 0) & (df['num_schemes'] > 0)]
df.to_csv(os.path.join(processed_dir, "03_aum_by_fund_house_cleaned.csv"), index=False)
print("Cleaned AUM by Fund House saved.")

## Part 4: Monthly SIP Inflows

Cleaning steps for `04_monthly_sip_inflows.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "04_monthly_sip_inflows.csv"))
# Convert YYYY-MM to first of the month
df['month'] = pd.to_datetime(df['month'] + '-01')
df = df[df['sip_inflow_crore'] > 0]
df.to_csv(os.path.join(processed_dir, "04_monthly_sip_inflows_cleaned.csv"), index=False)
print("Cleaned Monthly SIP Inflows saved.")

## Part 5: Category Inflows

Cleaning steps for `05_category_inflows.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "05_category_inflows.csv"))
df['month'] = pd.to_datetime(df['month'] + '-01')
df['category'] = df['category'].str.strip()
df.to_csv(os.path.join(processed_dir, "05_category_inflows_cleaned.csv"), index=False)
print("Cleaned Category Inflows saved.")

## Part 6: Industry Folio Count

Cleaning steps for `06_industry_folio_count.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "06_industry_folio_count.csv"))
df['month'] = pd.to_datetime(df['month'] + '-01')
df = df[df['total_folios_crore'] > 0]
df.to_csv(os.path.join(processed_dir, "06_industry_folio_count_cleaned.csv"), index=False)
print("Cleaned Industry Folio Count saved.")

## Part 7: Scheme Performance

Cleaning steps for `07_scheme_performance.csv`.

In [ ]:
perf_df = pd.read_csv(os.path.join(raw_dir, "07_scheme_performance.csv"))
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    perf_df[col] = pd.to_numeric(perf_df[col], errors='coerce')
perf_df['expense_ratio_pct'] = pd.to_numeric(perf_df['expense_ratio_pct'], errors='coerce')
perf_df.to_csv(os.path.join(processed_dir, "07_scheme_performance_cleaned.csv"), index=False)
print("Cleaned Scheme Performance saved.")

## Part 8: Investor Transactions

Cleaning steps for `08_investor_transactions.csv`.

In [ ]:
trans_df = pd.read_csv(os.path.join(raw_dir, "08_investor_transactions.csv"))
trans_df['transaction_date'] = pd.to_datetime(trans_df['transaction_date'])
type_mapping = {'sip': 'SIP', 'lumpsum': 'Lumpsum', 'redemption': 'Redemption'}
trans_df['transaction_type'] = trans_df['transaction_type'].str.strip().str.lower().map(lambda x: type_mapping.get(x, x.capitalize()))
trans_df = trans_df[trans_df['amount_inr'] > 0]
trans_df['kyc_status'] = trans_df['kyc_status'].str.strip().str.capitalize()
trans_df.to_csv(os.path.join(processed_dir, "08_investor_transactions_cleaned.csv"), index=False)
print("Cleaned Investor Transactions saved.")

## Part 9: Portfolio Holdings

Cleaning steps for `09_portfolio_holdings.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "09_portfolio_holdings.csv"))
df['portfolio_date'] = pd.to_datetime(df['portfolio_date'])
df = df[df['weight_pct'] > 0]
df.to_csv(os.path.join(processed_dir, "09_portfolio_holdings_cleaned.csv"), index=False)
print("Cleaned Portfolio Holdings saved.")

## Part 10: Benchmark Indices

Cleaning steps for `10_benchmark_indices.csv`.

In [ ]:
df = pd.read_csv(os.path.join(raw_dir, "10_benchmark_indices.csv"))
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['index_name', 'date'])
df = df[df['close_value'] > 0]
df.to_csv(os.path.join(processed_dir, "10_benchmark_indices_cleaned.csv"), index=False)
print("Cleaned Benchmark Indices saved.")